In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import scipy.stats as stats

# Set visual style for the final paper
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12, 'figure.dpi': 150})

def load_algorithm_data(folder_path, algo_name):
    path = Path(folder_path)
    # This specifically targets ONLY .csv files, ignoring .sol or .vrp files!
    all_files = list(path.glob("*.csv")) 
    
    if not all_files:
        print(f"[Warning] No CSVs found in {folder_path}!")
        return pd.DataFrame()
        
    df_list = []
    for f in all_files:
        df = pd.read_csv(f)
        df['Algorithm'] = algo_name
        df_list.append(df)
        
    final_df = pd.concat(df_list, ignore_index=True)
    print(f"Loaded {len(final_df)} runs for {algo_name}.")
    return final_df

# --- Load the Champions ---
# Pointing directly to the winning sub-folders for a fair comparison
df_ga = load_algorithm_data("experiments/ga/feasibility_archive", "GA (Crowding)")
df_aco = load_algorithm_data("experiments/aco/feasibility_archive", "ACO")

# Change 'sc' to 'complete_cluster' if that was the PSO team's winning variant!
df_pso = load_algorithm_data("experiments/pso/sc", "PSO") 

# Combine into one massive global dataset
df_global = pd.concat([df_ga, df_aco, df_pso], ignore_index=True)

# Display a preview
display(df_global.head())

In [ ]:
# Group by Algorithm and calculate Mean & Std for the crucial metrics
global_summary = df_global.groupby('Algorithm').agg(
    Avg_Cost=('mean_fitness', 'mean'),
    Cost_Std=('mean_fitness', 'std'),
    Avg_Diversity=('mean_jaccard_distance', 'mean'),
    Avg_Time=('tiempo', 'mean')
).reset_index()

# Format the results cleanly
global_summary['Avg_Cost'] = global_summary['Avg_Cost'].round(2)
global_summary['Cost_Std'] = global_summary['Cost_Std'].round(2)
global_summary['Avg_Diversity'] = (global_summary['Avg_Diversity'] * 100).round(2).astype(str) + '%'
global_summary['Avg_Time'] = global_summary['Avg_Time'].round(2).astype(str) + 's'

print("=== GLOBAL PERFORMANCE SUMMARY ===")
display(global_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
palette = {"GA (Crowding)": "#2ecc71", "ACO": "#e74c3c", "PSO": "#3498db"}

# --- Plot 1: Routing Cost (Lower is Better) ---
sns.boxplot(data=df_global, x='Algorithm', y='mean_fitness', ax=axes[0], palette=palette)
axes[0].set_title("Global Routing Cost Distribution (Lower is Better)", fontweight='bold')
axes[0].set_ylabel("Routing Cost (Fitness)")
axes[0].set_xlabel("")

# --- Plot 2: Structural Diversity (Higher is Better) ---
sns.boxplot(data=df_global, x='Algorithm', y='mean_jaccard_distance', ax=axes[1], palette=palette)
axes[1].set_title("Global Multimodal Diversity (Higher is Better)", fontweight='bold')
axes[1].set_ylabel("Jaccard Distance")
axes[1].set_xlabel("")

plt.tight_layout()
plt.savefig("experiments/global_comparison_boxplots.png")
plt.show()